# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from training import *
import torch
from itertools import product


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 200

In [3]:
df = pd.read_csv(f"../../data/top30groups/anonLoc/combined/combined{partition}.csv")

In [4]:
from pathlib import Path
import numpy as np
from datetime import datetime

def save_metrics_txt(best_metrics, partition, best_params, append=False):
    """
    Write best_metrics to results/results_{partition}.txt as key: value lines.
    Set append=True to add another block instead of overwriting.
    """
    results_dir = Path("results")
    results_dir.mkdir(parents=True, exist_ok=True)
    path = results_dir / f"results_{partition}.txt"
    mode = "a" if append else "w"

    # make JSON-friendly scalars for numpy types
    def to_scalar(v):
        if isinstance(v, (np.floating,)):
            return float(v)
        if isinstance(v, (np.integer,)):
            return int(v)
        return v

    with open(path, mode, encoding="utf-8") as f:
        if append:
            f.write("\n" + "="*60 + "\n")
        f.write(f"Run saved: {datetime.now().isoformat(timespec='seconds')}\n")
        # align keys for readability
        f.write(f"{best_params}\n")
        width = max(len(k) for k in best_metrics.keys())
        for k in sorted(best_metrics.keys()):
            f.write(f"{k:<{width}} : {to_scalar(best_metrics[k])}\n")

# usage:
# save_metrics_txt(best_metrics, partition="gtd300", append=False)


In [6]:
import random 
node_feature_cols = [c for c in df.columns if c!='gname']
edge_feature_cols = ['attacktype1', 'target1']
edge_mode = 'hybrid_equal_knn'
equal_cols = ['encodedlonglat']
#k = 10
"""param_grid = {
    'hidden_dims': [32, 64, 128, 256],
    'dropouts_gcn': [0.1, 0.3, 0.5],
    'lr': [0.01],
    'n_tree': [50, 70],
    'tree_depth': [8,9,10],
    'tree_feature_rate': [0.1,0.3],
    'feat_dropout': [0.1,0.3],
    'out_size_nrf': [512,768]
    }
    sample 100 gave"""
    #{'hidden_dims': 256, 'dropouts_gcn': 0.1, 'lr': 0.01, 'n_tree': 70, 'tree_depth': 9, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'out_size_nrf': 512, 'epochs': 300, 'partition': 'gtd200', 'final_evaluation': False, 'n_class': 30}
#(256, 0.1, 0.01, 100, 9, 0.3, 0.3, 512)
#Best validation acc: 0.5408 @ epoch 295
#{'hidden_dims': 128, 'dropouts_gcn': 0.1, 'lr': 0.01, 'n_tree': 120, 'tree_depth': 9, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'out_size_nrf': 256, 'epochs': 300, 'partition': 'gtd200', 'final_evaluation': False, 'n_class': 30}

#{'hidden_dims': 128, 'dropouts_gcn': 0.1, 'lr': 0.01, 'n_tree': 120, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.3, 'out_size_nrf': 256, 'epochs': 400, 'partition': 'gtd200', 'final_evaluation': False, 'n_class': 30}
"""
param_grid = {
    'hidden_dims': [128],
    'dropouts_gcn': [0.1],
    'lr': [0.01],
    'n_tree': [80,100,120],
    'tree_depth': [8,9,10,11],
    'tree_feature_rate': [0.1,0.3],
    'feat_dropout': [0.1,0.3],
    'out_size_nrf': [256,512]
    }"""

#{'hidden_dims': 128, 'dropouts_gcn': 0.1, 'lr': 0.01, 'n_tree': 130, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.3, 'out_size_nrf': 256, 'wd': 0.001, 'epochs': 300, 'partition': 'gtd200', 'final_evaluation': False, 'n_class': 30}
"""param_grid = {
    'hidden_dims': [128],
    'dropouts_gcn': [0.1],
    'lr': [0.01],
    'n_tree': [120, 130],
    'tree_depth': [7,8,9],
    'tree_feature_rate': [0.3],
    'feat_dropout': [0.3],
    'out_size_nrf': [256],
    'wd' : [0.001, 0.01, 0.1]
}"""
#(256, 0.1, 0.01, 120, 8, 0.3, 0.1, 256, 0.001)
#Best validation acc: 0.5467 @ epoch 295
"""param_grid = {
    'hidden_dims': [128, 256],
    'dropouts_gcn': [0.1,0.3,0.5],
    'lr': [0.01,0.001],
    'n_tree': [120,130,140],
    'tree_depth': [8,9,10],
    'tree_feature_rate': [0.1,0.3],
    'feat_dropout': [0.1,0.3],
    'out_size_nrf': [256,512],
    'wd' : [5e-4,0.001]
}"""
#(32, 0.1, 0.01, 50, 10, 0.1, 0.1, 768, 0.0005, 10)

param_grid = {
    'hidden_dims': [32],
    'dropouts_gcn': [0.1],
    'lr': [0.01],
    'n_tree': [50],
    'tree_depth': [10],
    'tree_feature_rate': [0.1],
    'feat_dropout': [0.1],
    'out_size_nrf': [768],
    'wd' : [5e-4],
    'ks' : [10]
}
    
grid_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())
#sampled_combos = random.sample(grid_combos, 100)

best_acc = -1
for combo in grid_combos:
    param_dict = dict(zip(param_names, combo))
    print(combo)
    nrf_cfg = {
        **dict(zip(param_names, combo)),
        "epochs": 3000,
        "partition": f"gtd{partition}",   # one of: "gtd100", "gtd200", "gtd300", "gtd478"
        "final_evaluation": True,
        "n_class": 30
    }

    test_acc, best_epoch, best_metrics, epoch_logs = train_joint_gcn_nrf(
        df,
        node_feature_cols,
        edge_feature_cols,
        edge_mode,
        equal_cols,
        param_dict['ks'],
        param_dict['hidden_dims'],
        param_dict['dropouts_gcn'],
        nrf_cfg,
        weight_decay=param_dict['wd'],
        device="cuda"
    )

    if test_acc > best_acc:
        best_acc = test_acc
        best_params = nrf_cfg

print(best_params)


(32, 0.1, 0.01, 50, 10, 0.1, 0.1, 768, 0.0005, 10)
Best test acc: 0.5458 @ epoch 2935
{'hidden_dims': 32, 'dropouts_gcn': 0.1, 'lr': 0.01, 'n_tree': 50, 'tree_depth': 10, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'out_size_nrf': 768, 'wd': 0.0005, 'ks': 10, 'epochs': 3000, 'partition': 'gtd200', 'final_evaluation': True, 'n_class': 30}


In [ ]:
print(hej)

NameError: name 'hej' is not defined

In [ ]:
save_metrics_txt(best_metrics, partition, best_params)